# CS 789 — Assignment 5: GraphRAG vs. LightRAG vs. Vector RAG — and Neo4j
**AI-Powered Cyber Attacks and Defenses · Fall 2026 · UNLV · Prof. Yoohwan Kim**

Same Vandelay Industries corpus and the same 26 questions as A4 — so your A4 numbers are the baseline.
This time the retriever is a **knowledge graph** that an LLM extracts from the text.

| Part | What you do | What you learn |
|---|---|---|
| A | Index with **Microsoft GraphRAG**; open its entities, relationships, **communities** and **community reports**; visualize; audit two reports; try **local / global / DRIFT / basic** search | what a GraphRAG index *is*, and where its summaries come from |
| B | Index the same corpus with **LightRAG**; compare the two graphs; its 5 query modes; **add a new incident** to both and time the update | MS GraphRAG vs. LightRAG — the design trade-off |
| C | Load the graph into **Neo4j**; write **Cypher** by hand; let **LangChain** write Cypher for you and watch where it fails | graph databases, text-to-Cypher, and why it is dangerous |
| D | **Scoreboard**: 26 questions × 9 pipelines, by question type, plus cost | where graphs win, where they lose, what they cost |

Everything runs on your Colab GPU: **Qwen3-4B-Instruct-2507** (chat) and **bge-base-en-v1.5** (embeddings),
both served by vLLM. No paid API. As in A4 you **build** with your browser AI assistant (🛠️ cells),
**measure** with `cs789rag.py`, and **think** in the ✍️ cells.

> Runtime: **Colab Pro, L4 GPU** (A100 is faster for indexing). Plan ~2 h of your time plus ~30 min of unattended runs.

---
## Part 0 — Setup (≈ 15 min)

`Runtime → Change runtime type → L4 GPU` (or A100 — indexing is about 3× faster). Run 0.1: it downloads the
files and installs GraphRAG, LightRAG, Neo4j and LangChain packages. GraphRAG needs **pandas 3**, newer than
Colab's, so the cell then **restarts the Python session on purpose** ("Your session crashed" is expected).
Run 0.1 **again**: it installs vLLM into its own environment and starts **two** model servers — the chat
model on port 8000 and the embedding model on port 8001 (about 6 minutes).

In [ ]:
# 0.1 Files, packages, and the two local model servers  (run it TWICE the first time — see the note below)
import os, sys, subprocess, time, re, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/A5"):
        !git clone -q https://github.com/unlv-cs789-f26/A5.git /content/A5
    %cd /content/A5
    if importlib.util.find_spec("graphrag") is None:
        !pip -q install -r requirements.txt
        print("
Packages installed. GraphRAG needs pandas 3, so Colab must restart the Python session now.
"
              "When it says 'Your session crashed / restarted', just run this cell again (or Runtime > Run all).")
        os.kill(os.getpid(), 9)          # restart the kernel so the new pandas/numpy/pyarrow are loaded
import pandas as pd
sys.path.insert(0, os.getcwd())
import cs789rag as H
H.install_vllm()
if not H.server_up(8001):                # after a session restart the servers are still running: reuse them
    H.start_embedder("BAAI/bge-base-en-v1.5", port=8001, gpu_mem=0.10)
if not H.server_up(8000):
    H.start_vllm("fp8", max_model_len=32768, gpu_mem=0.72)   # fp8 KV cache: A4 showed it is free, and it doubles capacity
print(H.llm("Name the MITRE ATT&CK technique ID for LSASS memory credential dumping. One line."))
docs = H.load_corpus(); QUESTIONS = H.load_questions()
print(len(docs), "documents,", len(QUESTIONS), "questions;  embedding dim =", H.embed(["test"]).shape[1])

✍️ **Deliverable 0**: screenshot of the output of 0.1 (both servers up, and the model's one-line answer —
is it right? The correct answer is T1003.001).

---
## Part A — Microsoft GraphRAG (≈ 35 min, of which indexing is ~15–20 min on an L4, ~7 on an A100)

GraphRAG (lecture 5-1/5-2) runs a pipeline over the corpus: chunk → the LLM extracts **entities** and
**relationships** from every chunk (plus one "gleaning" pass for anything it missed) → descriptions of the
same entity are merged and summarized → the graph is clustered into a hierarchy of **communities** with the
Leiden algorithm → the LLM writes a **community report** for every community. Local search starts from
entities; global search map-reduces over community reports.

The entity types are *your* choice and they matter. The default is `organization, person, geo, event`. For
a SOC corpus we use: host, account, group, software, vulnerability, malware, threat_actor, ip_address,
domain, attack_technique, incident, network_segment, organization, person.

In [ ]:
# A.1 Create the project and index it (provided). Prints a progress line every 20 s. Watch the time.
GR = H.graphrag_project("/content/gr_security", entity_types=H.SECURITY_ENTITY_TYPES)
!cat /content/gr_security/settings.yaml | head -40
gr_run = H.graphrag_index(GR)
T = H.graphrag_tables(GR)
for name, df in T.items():
    print(f"{name:18s} {df.shape}")

In [ ]:
# A.2 What did the LLM extract?
ent, rel = T["entities"], T["relationships"]
display(ent["type"].value_counts().to_frame("entities by type"))
display(ent.sort_values("degree", ascending=False)[["title", "type", "degree", "description"]].head(15))
display(rel.sort_values("weight", ascending=False)[["source", "target", "description", "weight"]].head(10))

✍️ **Questions A-1, A-2**

* **A-1** How long did indexing take, how many entities, relationships and communities did it produce, and
  how many LLM calls (see `gr_run["stats"]` or `logs/`)? Compare with embedding the same corpus in A4 (seconds).
* **A-2** Find **three extraction errors** in `ent`: a duplicate (the same host or person under two names), a
  wrong type, or something that is not an entity at all. Show the rows. Why do these errors matter for multi-hop questions?

#### 🛠️ A.3 Visualize the graph by community

**Contract** (the checks below depend on these exact names): `show_graph(T, level=0, min_degree=1, html='graph.html')`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook, T is a dict of pandas DataFrames from a Microsoft GraphRAG index:
T["entities"] has columns title, type, description, degree; T["relationships"] has source, target,
description, weight; T["communities"] has community, level, title, entity_ids (list of entity ids),
and T["entities"] has an "id" column matching those entity_ids.
Write a function show_graph(T, level=0, min_degree=1, html="graph.html") that:
- builds a networkx graph of the entities with degree >= min_degree and the relationships between them
- colors every node by its community at the given level (entities not in a community are grey)
- sizes nodes by degree and shows title, type and the first 200 chars of description on hover
- renders it with pyvis (notebook=True, cdn_resources="in_line", height 650px) and displays it in the
  notebook with IPython.display.HTML(open(html).read())
Then call show_graph(T, level=0). Put the code in the empty cell under "A.3" and run it.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: show_graph(T, level=0, min_degree=1, html='graph.html')

**A.4 Community reports.** Every community gets an LLM-written report: a title, a summary, an impact rating
and findings. Global search answers *only* from these reports — so a wrong sentence in a report becomes a
confident wrong answer. Run the cell: it prints the two highest-ranked reports and, for each, the text units
(source chunks) of its entities, so you can check the report against the source.

In [ ]:
rep = T["community_reports"].sort_values("rank", ascending=False)
display(rep[["community", "level", "title", "rank", "size"]].head(10))
tu = T["text_units"].set_index("id")
for _, r in rep.head(2).iterrows():
    print("=" * 100, f"\nCOMMUNITY {r['community']} (level {r['level']}): {r['title']}\n")
    print(r["full_content"][:3000])
    com = T["communities"].set_index("community").loc[r["community"]]
    unit_ids = list(com["text_unit_ids"])[:4]
    print("\n--- source text units (first 4) ---")
    for u in unit_ids:
        print(f"[{u[:8]}] {tu.loc[u, 'text'][:700]}\n")

✍️ **Question A-3 (community audit — do this yourself, no AI).** For each of the two reports, take every
*finding* and mark it **S** (supported by the source text), **U** (unsupported — not in the sources) or **X**
(contradicted). Give a table *report · finding · S/U/X · evidence (doc id or quote)*. How many findings
per report were not supported? Would you let an analyst act on these reports without the sources?

In [ ]:
# A.5 Four questions through GraphRAG's search methods (provided; ~5 min on an L4)
# DRIFT runs only on MH-01: it makes dozens of LLM calls per question (1-3 minutes each on a Colab GPU).
SAMPLE = ["MH-01", "AGG-01", "SH-02", "NEG-01"]
qs = {q["id"]: q for q in QUESTIONS}
for qid in SAMPLE:
    print("#" * 100, f"\n{qid} [{qs[qid]['type']}] {qs[qid]['question']}\n   gold: {qs[qid]['gold_answer']}")
    for method in (["local", "global", "drift", "basic"] if qid == "MH-01" else ["local", "global", "basic"]):
        t = time.time()
        ans = H.graphrag_ask(GR, method, qs[qid]["question"])
        print(f"\n--- {method} ({time.time() - t:.0f}s) [{H.score_answer(qs[qid], ans)['grade']}]\n{ans[:900]}")

✍️ **Question A-4** Which method answered the multi-hop question (MH-01) and the aggregation question
(AGG-01) correctly? Which was fastest and slowest? Explain *why* global search is good at "what are the
main themes" and bad at "which host …" (think about what its context contains).

---
## Part B — LightRAG (≈ 30 min)

LightRAG (HKU, 2024; lecture 5-1) also extracts entities and relationships with the LLM, but it does
**not** build communities or community reports. At query time the LLM pulls **low-level keywords**
(entities) and **high-level keywords** (themes) out of the question and matches them against the entity
and relationship vectors. Modes: `naive` (plain vector RAG over chunks), `local` (entities), `global`
(relationships), `hybrid` (both), `mix` (graph + chunks). Because there are no communities to rebuild,
adding a document is cheap — you will test that claim.

#### 🛠️ B.1 Build a LightRAG index on the local servers

**Contract** (the checks below depend on these exact names): `rag` (initialized LightRAG with all 35 docs), `lr_insert_sec`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
I am in a Google Colab notebook. Two OpenAI-compatible servers run locally: chat model "llm" at
http://localhost:8000/v1 and embedding model "emb" (768 dimensions) at http://localhost:8001/v1; the
api key is "EMPTY". There is a helper H.embed(list_of_texts) that returns a normalized numpy array.
Using lightrag-hku (from lightrag import LightRAG, QueryParam; from lightrag.llm.openai import
openai_complete_if_cache; from lightrag.utils import EmbeddingFunc; from lightrag.kg.shared_storage
import initialize_pipeline_status):
- write async llm_func(prompt, system_prompt=None, history_messages=None, keyword_extraction=False, **kwargs)
  that calls openai_complete_if_cache("llm", prompt, system_prompt=..., history_messages=history_messages or [],
  base_url=..., api_key="EMPTY", **kwargs)
- write async embed_func(texts) that returns H.embed(texts), wrapped as
  EmbeddingFunc(embedding_dim=768, max_token_size=512, func=embed_func)
- create rag = LightRAG(working_dir="/content/lightrag_store", llm_model_func=llm_func,
  embedding_func=..., llm_model_max_async=32, max_parallel_insert=8) and initialize it with await rag.initialize_storages()
  and await initialize_pipeline_status()
- insert every document d in docs with await rag.ainsert([d.text ...], ids=[d.doc_id ...],
  file_paths=[d.doc_id ...]) and print how many seconds the insert took (store it in lr_insert_sec)
Use top-level await (the notebook supports it). Put it in the empty cell under "B.1" and run it.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: rag (initialized LightRAG with all 35 docs), lr_insert_sec

In [ ]:
# B.2 Compare the two graphs (provided)
import networkx as nx, glob
G_lr = nx.read_graphml(glob.glob("/content/lightrag_store/**/graph_chunk_entity_relation.graphml", recursive=True)[0])
lr_types = pd.Series([d.get("entity_type", "?") for _, d in G_lr.nodes(data=True)]).value_counts()
cmp = pd.DataFrame({
    "MS GraphRAG": [len(T["entities"]), len(T["relationships"]), len(T["communities"]), len(T["community_reports"]), gr_run["seconds"]],
    "LightRAG": [G_lr.number_of_nodes(), G_lr.number_of_edges(), 0, 0, lr_insert_sec]},
    index=["entities", "relationships", "communities", "community reports", "index seconds"])
display(cmp); display(lr_types.to_frame("LightRAG entities by type"))
# entities that both graphs found, and entities only one found
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
gr_names, lr_names = {norm(x) for x in T["entities"]["title"]}, {norm(x) for x in G_lr.nodes}
print(f"in both: {len(gr_names & lr_names)}   only GraphRAG: {len(gr_names - lr_names)}   only LightRAG: {len(lr_names - gr_names)}")
print("hosts found by both:", sorted(x for x in gr_names & lr_names if x.startswith("van")))

In [ ]:
# B.3 The five LightRAG modes on the same four questions (provided; ~2 min)
for qid in SAMPLE:
    print("#" * 100, f"\n{qid} {qs[qid]['question']}\n   gold: {qs[qid]['gold_answer']}")
    for mode in ["naive", "local", "global", "hybrid", "mix"]:
        t = time.time()
        ans = await rag.aquery(qs[qid]["question"], param=QueryParam(mode=mode, response_type="Single Paragraph"))
        print(f"\n--- {mode} ({time.time() - t:.0f}s) [{H.score_answer(qs[qid], ans)['grade']}]\n{str(ans)[:700]}")

In [ ]:
# What did LightRAG actually retrieve for MH-01?  (context only, no answer)
ctx = await rag.aquery(qs["MH-01"]["question"], param=QueryParam(mode="mix", only_need_context=True))
print(str(ctx)[:4000])

**B.4 A new incident arrives.** `new_ticket_inc_2026_049.md` is tomorrow's ticket. Add it to **both**
indexes and time it. For GraphRAG this is `graphrag update --method standard` (it re-runs extraction
for the new document and then rebuilds the affected communities and reports).

In [ ]:
NEW_TICKET = str(H.HERE / "new_ticket_inc_2026_049.md")    # absolute path: LightRAG may change the working directory
new_doc = open(NEW_TICKET).read()
t = time.time()
await rag.ainsert([new_doc], ids=["inc-2026-049"], file_paths=["inc-2026-049"])
lr_update_sec = round(time.time() - t, 1)
H.graphrag_project(GR, extra_files=[NEW_TICKET])
gr_update = H.graphrag_index(GR, method="standard-update")
print(f"LightRAG update: {lr_update_sec}s    GraphRAG standard-update: {gr_update['seconds']}s")
UPD_Q = "Which host made the blocked cloud upload attempt in INC-2026-049, and who is that host's primary owner?"
print("\nLightRAG mix :", await rag.aquery(UPD_Q, param=QueryParam(mode="mix", response_type="Single Paragraph")))
print("\nGraphRAG local:", H.graphrag_ask(GR, "local", UPD_Q))

✍️ **Questions B-1 … B-4**

* **B-1** Fill *entities · relationships · communities · reports · index seconds* for both systems. Which
  graph is bigger, and why (think: gleaning, description summarization, community reports)?
* **B-2** Pick one entity both systems found and compare the two descriptions. Which is more useful to an analyst?
* **B-3** Which LightRAG mode was best on MH-01 and on AGG-01? What in the retrieved context (B.3 second cell)
  explains it?
* **B-4** The update: how many seconds each, and did each system answer the new question (VAN-SRV-BLD02,
  owned by Priya Raman)? For a SOC that receives 500 new tickets a day, which design would you choose and why?
  (Lecture 5-1: "The Problem with Standard GraphRAG".)

---
## Part C — Neo4j and Cypher (≈ 25 min)

**C.0 Get a Neo4j database.** *Option 1 (recommended, has the visual Explorer):* **Neo4j AuraDB Free** —
console.neo4j.io → sign in → *Create instance → AuraDB Free* → **download the credentials file** (the
password is shown only once) → wait until *Running*. Put `NEO4J_URI` (neo4j+s://xxxx.databases.neo4j.io),
`NEO4J_USERNAME` and `NEO4J_PASSWORD` into Colab 🔑 Secrets. *Option 2 (no account):* run
`!bash install_neo4j_colab.sh` — Neo4j Community inside this runtime (no visual Explorer; it is gone when the
runtime ends).

In [ ]:
# C.0 Choose your database. AuraDB (recommended): leave False and put the three secrets in Colab Secrets.
USE_LOCAL_NEO4J = False

In [ ]:
# C.0 Connect (provided)
from neo4j import GraphDatabase
if USE_LOCAL_NEO4J:
    !bash {H.HERE}/install_neo4j_colab.sh
    NEO4J = {"NEO4J_URI": "bolt://localhost:7687", "NEO4J_USERNAME": "neo4j", "NEO4J_PASSWORD": "cs789-a5-local"}
else:
    from google.colab import userdata
    NEO4J = {k: userdata.get(k) for k in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")}
driver = GraphDatabase.driver(NEO4J["NEO4J_URI"], auth=(NEO4J["NEO4J_USERNAME"], NEO4J["NEO4J_PASSWORD"]))
driver.verify_connectivity()
print("connected to", NEO4J["NEO4J_URI"])

#### 🛠️ C.1 Load the LightRAG graph into Neo4j

**Contract** (the checks below depend on these exact names): `load_graph(driver, G)`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook, G_lr is a networkx graph read from LightRAG's graphml file: node ids are entity
names, node attributes include entity_type and description; edge attributes include description,
keywords and weight. driver is a connected neo4j.GraphDatabase driver.
Write load_graph(driver, G) that:
- deletes everything in the database first (MATCH (n) DETACH DELETE n)
- creates a uniqueness constraint on :Entity(name) if it does not exist
- loads all nodes as (:Entity {name, type, description}) with UNWIND batches of 500
- loads all edges as (:Entity)-[:RELATED {description, keywords, weight}]->(:Entity), in batches, using MERGE on the endpoints
- also adds a second label from the entity type, e.g. :Host or :Person (sanitize it to letters only,
  capitalized), using apoc-free Cypher (loop in Python per type if needed)
- prints the node and relationship counts from the database at the end
Call load_graph(driver, G_lr). Put it in the empty cell under "C.1" and run it.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: load_graph(driver, G)

In [ ]:
def cypher(q, **params):
    with driver.session() as s:
        return pd.DataFrame([r.data() for r in s.run(q, **params)])
cypher("MATCH (e:Entity) RETURN e.type AS type, count(*) AS n ORDER BY n DESC LIMIT 15")

✍️ **C.2 Write these three Cypher queries yourself** (no AI for the first attempt — Cypher is on the
midterm; lecture 4-2 "Cypher Query Language"). Run each with `cypher("...")` in the cells below and paste
query + result into your report. Entity names are stored as LightRAG extracted them (look them up first with
`cypher("MATCH (e:Entity) WHERE e.name CONTAINS 'WKS' RETURN e.name, e.type")`).

1. Everything directly connected to the workstation **VAN-WKS-4471** (neighbor name, type, relationship description).
2. The **shortest path** from the incident **INC-2026-041** to the **Meridian Payroll DB** (use `shortestPath`, up to 6 hops). Is it the planted chain ticket → host → owner → group → database?
3. The 10 entities with the **highest degree**, with their type. Compare with the top of `T["entities"]` from GraphRAG.

In [ ]:
# C.2 query 1
cypher("""

""")

In [ ]:
# C.2 query 2
cypher("""

""")

In [ ]:
# C.2 query 3
cypher("""

""")

**C.3 Text-to-Cypher with LangChain.** `GraphCypherQAChain` shows the LLM the graph schema, asks it to
*write* a Cypher query for your question, runs the query, and phrases the result. Run the cell: it prints the
generated Cypher for four questions.

> 🔐 Note the argument `allow_dangerous_requests=True`. The LLM writes a database query from user text: a
> prompt-injected question can produce `DETACH DELETE`. In production the chain must use a **read-only**
> database user. That is the same lesson as SQL injection — one level up.

In [ ]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI
graph = Neo4jGraph(url=NEO4J["NEO4J_URI"], username=NEO4J["NEO4J_USERNAME"], password=NEO4J["NEO4J_PASSWORD"],
                   enhanced_schema=False)
chat = ChatOpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY", model="llm", temperature=0)
chain = GraphCypherQAChain.from_llm(chat, graph=graph, verbose=False, return_intermediate_steps=True,
                                    allow_dangerous_requests=True, top_k=20)
for qid in ["SH-01", "MH-01", "AGG-01", "MH-03"]:
    q = qs[qid]["question"]
    try:
        out = chain.invoke({"query": q})
        cy = out["intermediate_steps"][0]["query"]; ctx = out["intermediate_steps"][1]["context"]
        print(f"{'#' * 90}\n{qid}: {q}\nCYPHER : {cy}\nROWS   : {str(ctx)[:300]}\nANSWER : {out['result']}")
    except Exception as e:
        print(f"{'#' * 90}\n{qid}: {q}\nFAILED : {str(e)[:300]}")

✍️ **Questions C-1 … C-3**

* **C-1** Paste your three Cypher queries and results. Did the shortest path follow the planted chain? If not,
  which link was missing or wrong in the extracted graph?
* **C-2** For the four LangChain questions: which generated Cypher was correct, which ran but returned
  nothing, which failed? Classify each failure: wrong entity name, wrong relationship direction/type, wrong
  property, or impossible from this graph.
* **C-3** Take a screenshot of the MH-01 chain in Neo4j (AuraDB: *Explore* → search the host name → expand;
  local: the pyvis view of the path). What would an analyst gain from this view that a paragraph of text does not give?

---
## Part D — The scoreboard (≈ 20 min, mostly unattended)

All 26 questions through nine pipelines, scored exactly as in A4. The vector baseline is A4's hybrid
(BM25 + bge-base, recursive 800-char chunks, k = 8), rebuilt here so every pipeline uses the same LLM.

In [ ]:
# D.1 Vector baseline, rebuilt (provided)
import re, numpy as np
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
split = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
CH = [{"doc_id": d.doc_id, "chunk_id": f"{d.doc_id}::{i}", "title": d.title, "text": t}
      for d in docs for i, t in enumerate(split.split_text(d.text))]
id_tok = lambda s: re.findall(r"[a-z0-9]+(?:[-_./:][a-z0-9]+)*", s.lower())
bm = BM25Okapi([id_tok(c["text"]) for c in CH])
E = H.embed([c["text"] for c in CH])
QP = "Represent this sentence for searching relevant passages: "
def hybrid(q, k, depth=50):
    b = np.argsort(-bm.get_scores(id_tok(q)))[:depth]; d = np.argsort(-(E @ H.embed([QP + q])[0]))[:depth]
    s = {}
    for lst in (b, d):
        for r, i in enumerate(lst, 1): s[i] = s.get(i, 0) + 1 / (60 + r)
    return [CH[i] for i in sorted(s, key=s.get, reverse=True)[:k]]
SYSTEM = ("You are a SOC analyst assistant for Vandelay Industries. Answer ONLY from the documents provided. "
          "Cite the document ids you used in [square brackets]. Keep the answer under 120 words. "
          "If the documents do not contain the answer, reply exactly: I do not know.")
def vector_answer(q):
    hits = hybrid(q, 8)
    ctx = "\n\n".join(f"[{h['doc_id']}] {h['title']}\n{h['text']}" for h in hits)
    return H.llm(f"Documents:\n{ctx}\n\nQuestion: {q}\nAnswer:", system=SYSTEM), hits
H.check_retriever(hybrid, "hybrid baseline")

In [ ]:
# D.2 Run everything (provided). ~15-25 min on an L4: leave it running.
# LightRAG answers are computed first with top-level await (LightRAG is async), then scored.
LR_MODES = ["naive", "local", "global", "hybrid", "mix"]
LR_ANS, LR_SEC = {}, {}
for mode in LR_MODES:
    t = time.time()
    LR_ANS[mode] = {q["question"]: str(await rag.aquery(q["question"], param=QueryParam(mode=mode, response_type="Single Paragraph")))
                    for q in QUESTIONS}
    LR_SEC[mode] = round((time.time() - t) / len(QUESTIONS), 1)
PIPELINES = {
    "vector hybrid (A4 best)": vector_answer,
    "GraphRAG local":  lambda q: H.graphrag_ask(GR, "local", q),
    "GraphRAG global": lambda q: H.graphrag_ask(GR, "global", q),
    "GraphRAG basic":  lambda q: H.graphrag_ask(GR, "basic", q),
}
for mode in LR_MODES:
    PIPELINES[f"LightRAG {mode}"] = (lambda m: (lambda q: LR_ANS[m][q]))(mode)
for name, fn in PIPELINES.items():
    H.eval_answers("D-scoreboard", {"pipeline": name}, fn)
print("LightRAG seconds per question:", LR_SEC)
H.plot_bars("D-scoreboard", title="Answer accuracy by question type (26 questions)")
H.table("D-scoreboard")

✍️ **Questions D-1 … D-4**

* **D-1** Copy the scoreboard (9 pipelines × 6 types + overall + H + A + seconds). Put your A4 results for the
  same question types next to it. Which pipeline is best **overall**, and which is best on **multi_hop**,
  **aggregation** and **negation** separately?
* **D-2** Where did a graph pipeline **lose** to the vector baseline? Explain one loss by looking at the answer
  (`H.rows("D-scoreboard", "GraphRAG local")`) — is it an extraction error (Part A-2), a retrieval miss, or the LLM?
* **D-3** Cost table: *index seconds · update seconds · seconds per question · graph size* for vector, GraphRAG,
  LightRAG. At 10,000 documents instead of 35, which would you build first, and what would you *not* build?
* **D-4** GraphRAG vs. LightRAG in one paragraph, from your own numbers: communities and reports vs. keyword-driven
  retrieval; index cost; update cost; what each is best at. (Lecture 5-1 "LightRAG vs. MS LazyGraphRAG" is the frame.)

---
## Part E — Report (in the handout)

1. Tables: A-1, B-1, the D scoreboard, and the cost table.
2. Answers to A-1 … D-4 (short; numbers from *your* run). The A-3 community audit table is graded most heavily.
3. Screenshots: 0.1, the community graph (A.3), the Neo4j view of the MH-01 chain (C-3).
4. **AI-assistant log** as in A4: assistant used; three prompts (worked / failed / fix); one place its code ran but was wrong.
5. **Your recommendation**: you must add graph retrieval to the SOC RAG system you recommended in A4. Which
   system, which entity types, how you keep it current, and how you stop a poisoned ticket from becoming a
   "fact" in a community report.